In [1]:
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import joblib
import torch

# add path to project root
import sys
PROJECT_ROOT = Path.cwd()

if "notebooks" in PROJECT_ROOT.parts:
    PROJECT_ROOT = PROJECT_ROOT.parent
    
sys.path.append(str(PROJECT_ROOT))


from src.preprocess.gaming_text_preprocessor import GamingTextPreprocessor
from src.preprocess.gaming_text_labeler import GamingTextLabeler
from src.model.ensemble_model import EnsembleModel
from src.model.game_model_collection import GamingModelCollection
from src.model.social_media_collection import SocialMediaModelCollection
from src.model.bert_collection import BertToxicityModelCollection
from src.ensemble.score import ClassificationMetrics
from src.model.fixed_model_ensemble import FixedWeightEnsembleModel
from src.model.ensemble_model import EnsembleModel

# instantiate preprocessor and labeler
gaming_labeler = GamingTextLabeler()
metrics = ClassificationMetrics()

In [2]:
CONFIG = {
    'seed': 7524,
    'data_dir': PROJECT_ROOT / 'data' / 'processed_data',
    'model_dir': PROJECT_ROOT / 'models',
    'text_col': 'message',
    'label_col': 'label'
}

np.random.seed(CONFIG['seed'])
seed = CONFIG['seed']
tc, lc = CONFIG['text_col'], CONFIG['label_col']

In [3]:
# social media
social_binary_paths = list((CONFIG['model_dir'] / 'binary' ).glob('social_media_*.joblib'))
scaler_binary_path = CONFIG['model_dir'] / 'helper' / 'social_media_max_abs_scaler.joblib'
nb_tfidf_binary_path = CONFIG['model_dir'] / 'helper' / 'social_media_nb_tfidf.joblib'

# order by alphanumeric order to ensure consistent ordering of models and weights
social_binary_paths = sorted(social_binary_paths, key=lambda p: p.name)


social_multi_paths = list((CONFIG['model_dir'] / 'multi' ).glob('social_media_*.joblib'))
social_multi_paths = sorted(social_multi_paths, key=lambda p: p.name)
scaler_multi_path = CONFIG['model_dir'] / 'helper' / 'multi_max_abs_scaler.joblib'
nb_tfidf_multi_path = CONFIG['model_dir'] / 'helper' / 'multi_word_tfidf_vectorizer.joblib'

# gaming
gaming_binary_dir = CONFIG['model_dir'] / "binary"
gaming_binary_model_paths = list(gaming_binary_dir.glob("gaming_all_*.joblib"))


gaming_multi_dir = CONFIG['model_dir'] / "multi"
gaming_multi_model_paths = list(gaming_multi_dir.glob("gaming_all_*.joblib"))

# Load Data

In [4]:
# load WOT train data
d = CONFIG['data_dir']

wot_train = pd.read_parquet(d / 'wot' / 'x_train.parquet')
wot_train["data_source"] = "wot"

# load DOTA training data
dota_train = pd.read_parquet(d / 'dota' / 'x_train.parquet')
dota_train["data_source"] = "dota"


# combine datasets
train = pd.concat([wot_train, dota_train], ignore_index=True)

In [5]:
wot_val = pd.read_parquet(d / 'wot' / 'x_validation.parquet')
wot_val["data_source"] = "wot"
dota_val = pd.read_parquet(d / 'dota' / 'x_validation.parquet')
dota_val["data_source"] = "dota"

# combine holdout datasets
val = pd.concat([wot_val, dota_val], ignore_index=True)

# Prepare Objects


In [6]:
# instantiate preprocessor and labeler
gaming_preprocessor = GamingTextPreprocessor()
gaming_labeler = GamingTextLabeler()
metrics = ClassificationMetrics()

gaming_preprocessor.fit(train, text_column=tc)

In [7]:
# create 3-class
train = gaming_labeler.convert_three_class(
    train, 
    label_column=lc, 
    output_column='label_3class',
    data_source_column='data_source'
)
val = gaming_labeler.convert_three_class(
    val, 
    label_column=lc, 
    output_column='label_3class',
    data_source_column='data_source'
)

# create binary
train = gaming_labeler.convert_binary(
    train, 
    label_column=lc, 
    output_column='label_binary',
)

val = gaming_labeler.convert_binary(
    val, 
    label_column=lc, 
    output_column='label_binary',
)


In [8]:
# create gaming multi collection 

gaming_multi_model_collection = GamingModelCollection(
    model_joblibs = gaming_multi_model_paths
)

model_names = [
    'wot_Logistic_Regression',
    'wot_LinearSVC',
    'wot_Logistic_Regression_+_Calibrated(isotonic,_ensemble=False)',
    'wot_LinearSVC_+_Calibrated(isotonic,_ensemble=False)',
    'dota_Logistic_Regression',
    'dota_LinearSVC',
    'dota_Logistic_Regression_+_Calibrated(sigmoid,_ensemble=True)',
    'dota_LinearSVC_+_Calibrated(sigmoid,_ensemble=True)',
    'combined_Logistic_Regression',
    'combined_LinearSVC',
    'combined_Logistic_Regression_+_Calibrated(sigmoid,_ensemble=False)',
    'combined_LinearSVC_+_Calibrated(isotonic,_ensemble=True)'
]

weights = np.array([
    0.01500173, 0.06092781, 0.00230242, 0.04228979,
    0.04470104, 0.05293533, 0.09912261, 0.00050821,
    0.35757238, 0.06847398, 0.22296926, 0.03319543
])

threshold = np.float64(0.7000000000000002)

weights_dict = dict(zip(model_names, weights))


gaming_multi_ensemble = FixedWeightEnsembleModel(
    model_collections=[gaming_multi_model_collection],
    weights=weights_dict,
    threshold=threshold
)

# create gaming binary collection

gaming_binary_model_collection = GamingModelCollection(
    model_joblibs = gaming_binary_model_paths
)

model_names =  ['wot_Logistic_Regression',
    'wot_LinearSVC',
    'wot_Logistic_Regression_+_Calibrated(isotonic,_ensemble=True)',
    'wot_LinearSVC_+_Calibrated(isotonic,_ensemble=True)',
    'dota_Logistic_Regression',
    'dota_LinearSVC',
    'dota_Logistic_Regression_+_Calibrated(sigmoid,_ensemble=False)',
    'dota_LinearSVC_+_Calibrated(sigmoid,_ensemble=True)',
    'combined_Logistic_Regression',
    'combined_LinearSVC',
    'combined_Logistic_Regression_+_Calibrated(isotonic,_ensemble=True)',
    'combined_LinearSVC_+_Calibrated(sigmoid,_ensemble=False)'
]

gaming_binary_weights = [0.04523348, 0.00946077, 0.08420375, 0.03204597, 0.09418352,
        0.03581342, 0.00528644, 0.04582971, 0.49255951, 0.02053279,
        0.05121582, 0.08363481]

gaming_binary_threshold = 0.8000000000000003

weights_dict = dict(zip(model_names, gaming_binary_weights))

gaming_binary_ensemble = FixedWeightEnsembleModel(
    model_collections=[gaming_binary_model_collection],
    weights=weights_dict,
    threshold=gaming_binary_threshold
)


In [9]:
# it works thank GOD
gaming_multi_ensemble.predict(val[tc])

Constructing confidence tensor...
Total models in ensemble: 12
Expected confidence shape: (n_samples=17056, n_classes=3)
Ensuring weights from dictionary...
Model names and weight keys match. Constructing weight array...
Respecting order of model names...
Predicted labels shape: (17056,)


array([ 0,  1, -1, ..., -1,  0,  0], shape=(17056,))

In [10]:
# create social media binary collection
social_media_binary_collection = SocialMediaModelCollection(
    model_joblibs=social_binary_paths,
    scaler_path=scaler_binary_path,
    nb_tfidf_path=nb_tfidf_binary_path,
)

social_media_binary_weights = {
    "social_media_ComplementNB_model" : 0.01692044,
    "social_media_LinearSVC_(Optuna)_model" : 0.89785828,
    "social_media_LogisticRegressionCV_model" : 0.08522128
}
social_media_binary_threshold = 0.7000000000000002

social_media_binary_ensemble = FixedWeightEnsembleModel(
    model_collections=[social_media_binary_collection],
    weights=social_media_binary_weights,
    threshold=social_media_binary_threshold
)

In [ ]:
# create bert multi collection
bert_multi_collection = BertToxicityModelCollection(
    model_names=["dota_multi", "jigsaw_multi", "wot_multi"]
)

bert_multi_model_names = ['dota_multi', 'jigsaw_multi', 'wot_multi']
bert_multi_weights = [0.51393845, 0.00572898, 0.48033257]
bert_multi_threshold = 0.001

bert_weights = dict(zip(bert_multi_model_names, bert_multi_weights))

bert_multi_ensemble = FixedWeightEnsembleModel(
    model_collections=[bert_multi_collection],
    weights=bert_weights,
    threshold=bert_multi_threshold
)


# create bert binary collection
bert_binary_collection = BertToxicityModelCollection(
    model_names=["dota_binary", "jigsaw_binary", "wot_binary"]
)

bert_binary_model_names = ['dota_binary', 'jigsaw_binary', 'wot_binary']
bert_binary_weights = [0.53242287, 0.09282784, 0.37474929]
bert_binary_threshold = 0.9

bert_binary_weights_dict = dict(zip(bert_binary_model_names, bert_binary_weights))

bert_binary_ensemble = FixedWeightEnsembleModel(
    model_collections=[bert_binary_collection],
    weights=bert_binary_weights_dict,
    threshold=bert_binary_threshold
)


Loading dota_multi from jforward/bert-toxicity/dota_multi_model...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 7357.79it/s]


Loading jigsaw_multi from jforward/bert-toxicity/jigsaw_multi_model...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 7042.48it/s]


Loading wot_multi from jforward/bert-toxicity/wot_multi_model...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 7203.26it/s]


NameError: name 'TODO' is not defined

In [22]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

X_train = train[tc]
y_train = train["label_binary"]

res = []

skf = StratifiedKFold(
    n_splits=2,
    shuffle=True,
    random_state=42
)

weight_grid = np.arange(0.0, 1.01, 0.1)

for gaming_weight in weight_grid:
    for social_media_weight in weight_grid:
        bert_weight = 1.0 - gaming_weight - social_media_weight

        # skip invalid combinations
        if bert_weight < -1e-9:
            continue

        # clean floating-point weirdness
        bert_weight = round(bert_weight, 10)

        fold_metrics = []

        for fold_idx, (_, val_idx) in enumerate(skf.split(X_train, y_train)):
            X_fold_val = X_train.iloc[val_idx]
            y_fold_val = y_train.iloc[val_idx]

            binary_ensemble = EnsembleModel(
                model_ensembles=[
                    gaming_binary_ensemble,
                    social_media_binary_ensemble,
                    bert_binary_ensemble,
                ],
                weights=[
                    gaming_weight,
                    social_media_weight,
                    bert_weight,
                ],
                name="binary_ensemble"
            )

            y_pred = binary_ensemble.predict(X_fold_val)

            metric_result = metrics.metrics(y_pred, y_fold_val)
            fold_metrics.append(metric_result)

        avg_f1 = np.mean([m["CV Macro F1"] for m in fold_metrics])
        std_f1 = np.std([m["CV Macro F1"] for m in fold_metrics])

        avg_accuracy = (
            np.mean([m["Accuracy"] for m in fold_metrics])
            if "Accuracy" in fold_metrics[0]
            else None
        )

        avg_fpr = (
            np.mean([m["FPR"] for m in fold_metrics])
            if "FPR" in fold_metrics[0]
            else None
        )

        print(
            f"Gaming: {gaming_weight:.2f}, "
            f"Social: {social_media_weight:.2f}, "
            f"BERT: {bert_weight:.2f}, "
            f"CV Macro F1: {avg_f1:.4f} ± {std_f1:.4f}"
        )

        res.append({
            "gaming_weight": gaming_weight,
            "social_media_weight": social_media_weight,
            "bert_weight": bert_weight,
            "cv_macro_f1_mean": avg_f1,
            "cv_macro_f1_std": std_f1,
            "cv_accuracy_mean": avg_accuracy,
            "cv_fpr_mean": avg_fpr,
            "fold_metrics": fold_metrics,
        })

res_df = pd.DataFrame(res)

best_row_idx = res_df["cv_macro_f1_mean"].idxmax()
best_row = res_df.loc[best_row_idx]

best_gaming_weight = best_row["gaming_weight"]
best_social_media_weight = best_row["social_media_weight"]
best_bert_weight = best_row["bert_weight"]

print("\nBEST WEIGHTS")
print("Gaming:", best_gaming_weight)
print("Social Media:", best_social_media_weight)
print("BERT:", best_bert_weight)
print("Best CV Macro F1:", best_row["cv_macro_f1_mean"])

Constructing confidence tensor...
Total models in ensemble: 12
Expected confidence shape: (n_samples=26854, n_classes=2)
Ensuring weights from dictionary...
Model names and weight keys match. Constructing weight array...
Respecting order of model names...
Predicted labels shape: (26854,)
Constructing confidence tensor...
Total models in ensemble: 3
Expected confidence shape: (n_samples=26854, n_classes=2)
Ensuring weights from dictionary...
Model names and weight keys match. Constructing weight array...
Respecting order of model names...
Predicted labels shape: (26854,)
Constructing confidence tensor...
Total models in ensemble: 12
Expected confidence shape: (n_samples=26853, n_classes=2)
Ensuring weights from dictionary...
Model names and weight keys match. Constructing weight array...
Respecting order of model names...
Predicted labels shape: (26853,)
Constructing confidence tensor...
Total models in ensemble: 3
Expected confidence shape: (n_samples=26853, n_classes=2)
Ensuring weigh

In [23]:
res_df.sort_values(["cv_f1_mean", "cv_accuracy_mean"], ascending=[False, False], inplace=True)
res_df

,gaming_weight,social_media_weight,cv_f1_mean,cv_f1_std,fold_metrics,cv_accuracy_mean,cv_FPR_mean
6,0.6,0.4,0.574165,0.000233,"[{'CV Macro F1': 0.5743984640196813, 'CV Weigh...",0.851770,None
7,0.7,0.3,0.574165,0.000233,"[{'CV Macro F1': 0.5743984640196813, 'CV Weigh...",0.851770,None
8,0.8,0.2,0.574165,0.000233,"[{'CV Macro F1': 0.5743984640196813, 'CV Weigh...",0.851770,None
9,0.9,0.1,0.574165,0.000233,"[{'CV Macro F1': 0.5743984640196813, 'CV Weigh...",0.851770,None
10,1.0,0.0,0.574165,0.000233,"[{'CV Macro F1': 0.5743984640196813, 'CV Weigh...",0.851770,None
5,0.5,0.5,0.475430,0.000932,"[{'CV Macro F1': 0.47636165246918516, 'CV Weig...",0.662297,None
0,0.0,1.0,0.462460,0.001220,"[{'CV Macro F1': 0.46368034490120585, 'CV Weig...",0.679837,None
1,0.1,0.9,0.462460,0.001220,"[{'CV Macro F1': 0.46368034490120585, 'CV Weig...",0.679837,None
2,0.2,0.8,0.462460,0.001220,"[{'CV Macro F1': 0.46368034490120585, 'CV Weig...",0.679837,None
3,0.3,0.7,0.462460,0.001220,"[{'CV Macro F1': 0.46368034490120585, 'CV Weig...",0.679837,None


In [16]:
final_binary_ensemble = EnsembleModel(
    model_ensembles=[
        gaming_binary_ensemble,
        social_media_binary_ensemble,
        bert_binary_ensemble,
    ],
    weights=[
        best_gaming_weight,
        best_social_media_weight,
        best_bert_weight,
    ],
    name="final_binary_ensemble"
)

metrics.metrics(final_binary_ensemble.predict(val[tc]), val['label_binary'])

Constructing confidence tensor...
Total models in ensemble: 12
Expected confidence shape: (n_samples=17056, n_classes=2)
Ensuring weights from dictionary...
Model names and weight keys match. Constructing weight array...
Respecting order of model names...
Predicted labels shape: (17056,)
Constructing confidence tensor...
Total models in ensemble: 3
Expected confidence shape: (n_samples=17056, n_classes=2)
Ensuring weights from dictionary...
Model names and weight keys match. Constructing weight array...
Respecting order of model names...
Predicted labels shape: (17056,)


{'CV Macro F1': 0.5296666364276489,
 'CV Weighted F1': 0.7518579368024302,
 'Accuracy': 0.8047607879924953,
 'Coverage': np.float64(1.0),
 'Precision': 0.47737922691030876}

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

X_train = train[tc]
y_train = train["label_3class"]   # multiclass label

res = []

skf = StratifiedKFold(
    n_splits=2,
    shuffle=True,
    random_state=42
)

for gaming_weight in np.arange(0.0, 1.01, 0.1):
    bert_weight = 1.0 - gaming_weight

    fold_metrics = []

    for fold_idx, (_, val_idx) in enumerate(skf.split(X_train, y_train)):
        X_fold_val = X_train.iloc[val_idx]
        y_fold_val = y_train.iloc[val_idx]

        multi_ensemble = EnsembleModel(
            model_ensembles=[
                gaming_multi_ensemble,
                bert_multi_ensemble,
            ],
            weights=[
                gaming_weight,
                bert_weight,
            ],
            name="multi_ensemble"
        )

        y_pred = multi_ensemble.predict(X_fold_val)

        metric_result = metrics.metrics(y_pred, y_fold_val)
        fold_metrics.append(metric_result)

    avg_f1 = np.mean([m["CV Macro F1"] for m in fold_metrics])
    std_f1 = np.std([m["CV Macro F1"] for m in fold_metrics])

    avg_accuracy = (
        np.mean([m["Accuracy"] for m in fold_metrics])
        if "Accuracy" in fold_metrics[0]
        else None
    )

    avg_fpr = (
        np.mean([m["FPR"] for m in fold_metrics])
        if "FPR" in fold_metrics[0]
        else None
    )

    print(
        f"Gaming weight: {gaming_weight:.2f}, "
        f"BERT weight: {bert_weight:.2f}, "
        f"CV Macro F1: {avg_f1:.4f} +- {std_f1:.4f}"
    )

    res.append({
        "gaming_weight": gaming_weight,
        "bert_weight": bert_weight,
        "cv_f1_mean": avg_f1,
        "cv_f1_std": std_f1,
        "cv_accuracy_mean": avg_accuracy,
        "cv_FPR_mean": avg_fpr,
        "fold_metrics": fold_metrics,
    })

res_df = pd.DataFrame(res)

best_row_idx = res_df["cv_f1_mean"].idxmax()

best_gaming_weight = res_df.loc[best_row_idx, "gaming_weight"]
best_bert_weight = res_df.loc[best_row_idx, "bert_weight"]

print("Best gaming weight:", best_gaming_weight)
print("Best BERT weight:", best_bert_weight)
print("Best CV Macro F1:", res_df.loc[best_row_idx, "cv_f1_mean"])

In [ ]:
final_multi_ensemble = EnsembleModel(
    model_ensembles=[
        gaming_multi_ensemble,
        bert_multi_ensemble,
    ],
    weights=[
        best_gaming_weight,
        best_bert_weight,
    ],
    name="final_multi_ensemble"
)

metrics.metrics(final_multi_ensemble.predict(val[tc]), val['label_3class'])